# Appendix P — Tables 5 and 6: high-dimensional pushforward designs

This notebook reproduces the high-dimensional pushforward designs reported in Appendix P. Case 1 uses a high-dimensional Gaussian design with sparse outcome structure. Case 2 uses a high-dimensional Gaussian mixture with dense random features. For each case the notebook reports AME and finite-shift APE metrics for Data-SMR, Time-SMR, and Riesz regression.

In [ ]:

from pathlib import Path
import sys
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path.cwd()
for parent in [REPO, *REPO.parents]:
    if (parent / "src" / "genriesz" / "scorematchingriesz.py").exists():
        REPO = parent
        break
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import genriesz.scorematchingriesz as smr
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_SEED = 123
np.random.seed(RANDOM_SEED)
print("device:", DEVICE)


In [ ]:
N_TRIALS = 200
N = 1000
N_FOLDS = 2
P = 50
DELTA = 1.0
N_MC_TRUTH = 200000
HIDDEN_DIMS = (256, 256, 256)
OUTCOME_EPOCHS = 200
SCORE_STEPS = 4000
RATIO_STEPS = 4000
BATCH_SIZE = 256
INTEGRATION_STEPS = 200
DSM_SIGMA_MIN = 0.01
DSM_SIGMA_MAX = 1.0
SIGMA_EVAL = 0.01
AME_LOCAL_SHIFT = 0.05
CLIP_LOG_RATIO = 20.0
TABLE_TITLE = "Appendix P Tables 5 and 6: high-dimensional pushforward designs"
FIGURE_TITLE = "Appendix P: high-dimensional pushforward errors"

In [ ]:

def toeplitz_cov(p, rho=0.3):
    idx = np.arange(p)
    return rho ** np.abs(idx[:, None] - idx[None, :])


def sample_gaussian_x(n, p, seed, rho=0.3):
    rng = np.random.default_rng(seed)
    return rng.multivariate_normal(np.zeros(p), toeplitz_cov(p, rho), size=n).astype("float32")


def make_case1_mu(p=50, k=10, seed=0):
    rng = np.random.default_rng(seed)
    active = np.arange(1, 1 + k)
    nuisance = np.arange(1 + k, p)
    w_nuis = rng.normal(0, 0.05, size=len(nuisance))
    def mu(x):
        d = x[:, 0]
        za = x[:, active]
        zn = x[:, nuisance] if len(nuisance) else np.zeros((x.shape[0], 0))
        out = (1.0 + 0.8*d + 0.2*d**2 + 2.0*np.sin(d) + 0.5*za[:,0] + 0.5*np.tanh(za[:,1]) + 0.2*za[:,2]**2 + 0.1*za[:,3]**3 + d*za[:,0] + 0.3*d*np.sin(za[:,4]) + 0.05*za[:,5:].sum(axis=1))
        if zn.shape[1]:
            out = out + zn @ w_nuis
        return out
    return mu


def sample_case1(n, seed):
    x = sample_gaussian_x(n, P, seed, rho=0.3)
    mu = make_case1_mu(P, k=10, seed=RANDOM_SEED + 999)
    rng = np.random.default_rng(seed + 12345)
    y = mu(x) + rng.normal(size=n)
    return x.astype("float32"), y.astype("float32"), mu


def sample_case2(n, seed, rf_dim=200):
    rng = np.random.default_rng(seed)
    sigma = toeplitz_cov(P, 0.3)
    means=[]
    for k, dmean in enumerate(np.linspace(-3.0, 3.0, 4)):
        m=np.zeros(P); m[0]=dmean; m[1:6]=rng.normal(0,0.3,size=5); means.append(m)
    comps=rng.integers(0,4,size=n)
    x=np.zeros((n,P))
    for k in range(4):
        idx=np.where(comps==k)[0]
        if len(idx): x[idx]=rng.multivariate_normal(means[k], sigma, size=len(idx))
    W=rng.normal(size=(rf_dim,P)); b=rng.normal(size=rf_dim); v=rng.normal(size=rf_dim)
    def mu(xin):
        d=xin[:,0]
        return 1.0 + 0.5*d + 0.1*d**2 + np.sin(d) + 0.05*(np.tanh(xin @ W.T + b) @ v) + 0.2*d*np.tanh(xin[:,1])
    y=mu(x)+rng.normal(size=n)
    return x.astype("float32"), y.astype("float32"), mu


def truth_ame(mu, sampler, seed=0):
    x=sampler(N_MC_TRUTH, seed)
    eps=1e-3
    return float((mu(shift_x(x, eps)).mean() - mu(x).mean()) / eps)


def truth_ape(mu, sampler, delta, seed=0):
    x=sampler(N_MC_TRUTH, seed)
    return float(mu(shift_x(x, delta)).mean() - mu(shift_x(x, -delta)).mean())


In [ ]:

def shift_x(x, delta):
    out = np.asarray(x, dtype="float32").copy()
    out[:, 0] += float(delta)
    return out


def alpha_data_smr(x_train, x_eval, delta, seed):
    score_model = smr.fit_data_smr_score_dsm(x_train, hidden_dims=HIDDEN_DIMS, n_steps=SCORE_STEPS, batch_size=BATCH_SIZE, sigma_min=DSM_SIGMA_MIN, sigma_max=DSM_SIGMA_MAX, seed=seed, device=DEVICE)
    log_plus = smr.log_ratio_from_data_score_shift(score_model, x_eval, delta, steps=INTEGRATION_STEPS, sigma_eval=SIGMA_EVAL, direction="+", normalize=True, x_p_for_norm=x_train, device=DEVICE).reshape(-1)
    log_minus = smr.log_ratio_from_data_score_shift(score_model, x_eval, delta, steps=INTEGRATION_STEPS, sigma_eval=SIGMA_EVAL, direction="-", normalize=True, x_p_for_norm=x_train, device=DEVICE).reshape(-1)
    return np.exp(np.clip(log_plus, -CLIP_LOG_RATIO, CLIP_LOG_RATIO)) - np.exp(np.clip(log_minus, -CLIP_LOG_RATIO, CLIP_LOG_RATIO))


def alpha_time_smr_ratio(x_train, x_eval, delta, seed):
    m_plus = smr.fit_time_smr_dre_infinity(shift_x(x_train, delta), x_train, hidden_dims=HIDDEN_DIMS, n_steps=RATIO_STEPS, batch_size=BATCH_SIZE, seed=seed, device=DEVICE)
    m_minus = smr.fit_time_smr_dre_infinity(shift_x(x_train, -delta), x_train, hidden_dims=HIDDEN_DIMS, n_steps=RATIO_STEPS, batch_size=BATCH_SIZE, seed=seed+17, device=DEVICE)
    log_plus = smr.log_ratio_from_time_score(m_plus, x_eval, steps=INTEGRATION_STEPS, normalize=True, x_p_for_norm=x_train, device=DEVICE)
    log_minus = smr.log_ratio_from_time_score(m_minus, x_eval, steps=INTEGRATION_STEPS, normalize=True, x_p_for_norm=x_train, device=DEVICE)
    r_plus = np.exp(np.clip(log_plus.detach().cpu().numpy().reshape(-1), -CLIP_LOG_RATIO, CLIP_LOG_RATIO))
    r_minus = np.exp(np.clip(log_minus.detach().cpu().numpy().reshape(-1), -CLIP_LOG_RATIO, CLIP_LOG_RATIO))
    return r_plus - r_minus


def alpha_sq_riesz_ratio(x_train, x_eval, delta, seed):
    rm_plus = smr.fit_sq_riesz_ratio(shift_x(x_train, delta), x_train, hidden_dims=HIDDEN_DIMS, n_steps=RATIO_STEPS, batch_size=BATCH_SIZE, seed=seed, device=DEVICE)
    rm_minus = smr.fit_sq_riesz_ratio(shift_x(x_train, -delta), x_train, hidden_dims=HIDDEN_DIMS, n_steps=RATIO_STEPS, batch_size=BATCH_SIZE, seed=seed+17, device=DEVICE)
    r_plus = smr.eval_ratio_sq(rm_plus, x_eval, normalize=True, x_p_for_norm=x_train, device=DEVICE).reshape(-1)
    r_minus = smr.eval_ratio_sq(rm_minus, x_eval, normalize=True, x_p_for_norm=x_train, device=DEVICE).reshape(-1)
    return r_plus - r_minus


def alpha_time_smr_ame(x_train, x_eval, seed):
    return alpha_time_smr_ratio(x_train, x_eval, AME_LOCAL_SHIFT, seed) / (2.0 * AME_LOCAL_SHIFT)


def estimate_ame_trial(x, y, truth, seed):
    rows=[]
    for method in ["Data-SMR", "Time-SMR", "Riesz reg."]:
        scores=np.zeros(len(y))
        for train_idx, test_idx in smr.crossfit_splits(len(y), n_folds=N_FOLDS, seed=seed):
            x_train, y_train = x[train_idx], y[train_idx]
            x_test, y_test = x[test_idx], y[test_idx]
            outcome=smr.fit_outcome_net(x_train, y_train, hidden_dims=HIDDEN_DIMS, n_epochs=OUTCOME_EPOCHS, batch_size=BATCH_SIZE, seed=seed, device=DEVICE)
            gamma=smr.predict_outcome(outcome, x_test, device=DEVICE).reshape(-1)
            m_gamma=smr.partial_d_outcome(outcome, x_test, coordinate=0, device=DEVICE).reshape(-1)
            if method == "Data-SMR":
                score_model=smr.fit_data_smr_score_dsm(x_train, hidden_dims=HIDDEN_DIMS, n_steps=SCORE_STEPS, batch_size=BATCH_SIZE, sigma_min=DSM_SIGMA_MIN, sigma_max=DSM_SIGMA_MAX, seed=seed, device=DEVICE)
                alpha=-smr.eval_data_score_d(score_model, x_test, sigma_eval=SIGMA_EVAL, device=DEVICE).reshape(-1)
            elif method == "Time-SMR":
                alpha=alpha_time_smr_ame(x_train, x_test, seed)
            else:
                alpha_model=smr.fit_sq_riesz_ame(x_train, hidden_dims=HIDDEN_DIMS, n_steps=RATIO_STEPS, batch_size=BATCH_SIZE, seed=seed, device=DEVICE)
                alpha=smr.eval_scalar_net(alpha_model, x_test, device=DEVICE).reshape(-1)
            scores[test_idx] = m_gamma + alpha * (y_test - gamma)
        est=smr.wald_interval(scores)
        rows.append({"target":"AME", "method":method, "estimate":est.estimate, "se":est.se, "ci_low":est.ci_low, "ci_high":est.ci_high, "truth":truth, "error":est.estimate-truth, "covered":est.ci_low <= truth <= est.ci_high})
    return rows


def estimate_ape_trial(x, y, truth, delta, seed):
    rows=[]
    for method in ["Data-SMR", "Time-SMR", "Riesz reg."]:
        scores=np.zeros(len(y))
        for train_idx, test_idx in smr.crossfit_splits(len(y), n_folds=N_FOLDS, seed=seed):
            x_train, y_train = x[train_idx], y[train_idx]
            x_test, y_test = x[test_idx], y[test_idx]
            outcome=smr.fit_outcome_net(x_train, y_train, hidden_dims=HIDDEN_DIMS, n_epochs=OUTCOME_EPOCHS, batch_size=BATCH_SIZE, seed=seed, device=DEVICE)
            gamma=smr.predict_outcome(outcome, x_test, device=DEVICE).reshape(-1)
            m_gamma=smr.predict_outcome(outcome, shift_x(x_test, delta), device=DEVICE).reshape(-1) - smr.predict_outcome(outcome, shift_x(x_test, -delta), device=DEVICE).reshape(-1)
            if method == "Data-SMR":
                alpha=alpha_data_smr(x_train, x_test, delta, seed)
            elif method == "Time-SMR":
                alpha=alpha_time_smr_ratio(x_train, x_test, delta, seed)
            else:
                alpha=alpha_sq_riesz_ratio(x_train, x_test, delta, seed)
            scores[test_idx]=m_gamma + alpha * (y_test - gamma)
        est=smr.wald_interval(scores)
        rows.append({"target":"APE", "method":method, "estimate":est.estimate, "se":est.se, "ci_low":est.ci_low, "ci_high":est.ci_high, "truth":truth, "error":est.estimate-truth, "covered":est.ci_low <= truth <= est.ci_high})
    return rows


In [ ]:

def summarize_trials(df, group_cols):
    return df.groupby(group_cols).agg(
        trials=("estimate", "count"), truth=("truth", "mean"), bias=("error", "mean"),
        mse=("error", lambda s: float(np.mean(np.square(s)))), coverage=("covered", "mean"), avg_se=("se", "mean")
    ).reset_index()


In [ ]:

case_specs = [
    ("Case 1", sample_case1, lambda n, seed: sample_case1(n, seed)[0]),
    ("Case 2", sample_case2, lambda n, seed: sample_case2(n, seed)[0]),
]
rows=[]
for case_name, sampler_with_y, sampler_x in case_specs:
    _, _, mu = sampler_with_y(20, RANDOM_SEED)
    theta_ame = truth_ame(mu, sampler_x, seed=RANDOM_SEED + 500)
    theta_ape = truth_ape(mu, sampler_x, DELTA, seed=RANDOM_SEED + 501)
    for trial in range(N_TRIALS):
        x, y, _ = sampler_with_y(N, RANDOM_SEED + trial)
        for row in estimate_ame_trial(x, y, theta_ame, RANDOM_SEED + trial):
            rows.append({**row, "case": case_name, "trial": trial})
        for row in estimate_ape_trial(x, y, theta_ape, DELTA, RANDOM_SEED + trial):
            rows.append({**row, "case": case_name, "trial": trial})
appendix_p_results = pd.DataFrame(rows)
print(TABLE_TITLE)
display(summarize_trials(appendix_p_results, ["case", "target", "method"]))


In [ ]:

plot_df = appendix_p_results.copy()
methods = list(plot_df["method"].unique())
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, target in zip(axes, ["AME", "APE"]):
    sub = plot_df[plot_df["target"] == target]
    groups = [f"{c}\n{m}" for c in sub["case"].unique() for m in methods]
    data = [sub[(sub["case"] == c) & (sub["method"] == m)]["error"] for c in sub["case"].unique() for m in methods]
    ax.boxplot(data, labels=groups, showfliers=False)
    ax.axhline(0.0, linestyle="--")
    ax.set_title(f"{FIGURE_TITLE}: {target}")
    ax.tick_params(axis="x", rotation=60)
axes[0].set_ylabel("estimate minus truth")
fig.tight_layout()
plt.show()
